# Gated Attention in a small GPT (DL-058)

Paper: Qiu et al., *Gated Attention for Large Language Models* (NeurIPS 2025), arXiv:2505.06708

**Idea:** after attention, multiply its output by a gate `g = sigmoid(X · W_g)` (values between 0 and 1) before the output projection `W_o`.

**Plan:**
1. Train a small GPT with no gate (baseline) and with the gate in 3 places (G1, G2, G5).
2. Measure perplexity, the attention-sink score, and the average gate value.
3. Train baseline and G1 at high learning rates to compare stability.

Before running: **Runtime → Change runtime type → T4 GPU**, then **Run all** (about 40 minutes).

## Step 0: Install and import

In [ ]:
!pip -q install tiktoken datasets

import math, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Step 1: Settings

In [ ]:
D_MODEL  = 256      # size of each token vector
N_HEADS  = 4        # attention heads
N_LAYERS = 4        # transformer blocks
BLOCK    = 128      # tokens the model sees at once (context length)
BATCH    = 32       # sequences per training step
STEPS    = 2000     # training steps per run
WARMUP   = 100      # steps to slowly increase the learning rate
LR       = 1e-3     # normal learning rate
HIGH_LRS = [3e-3, 1e-2]   # high learning rates for the stability test

## Step 2: Load the data (TinyStories)

TinyStories is downloaded automatically from Hugging Face. We turn the text into token IDs with the GPT-2 tokenizer and put them in one long list.

In [ ]:
import tiktoken
from datasets import load_dataset

enc = tiktoken.get_encoding("gpt2")
VOCAB = enc.n_vocab   # 50257 possible tokens

def load_tokens(split, n_stories):
    stories = load_dataset("roneneldan/TinyStories", split=f"{split}[:{n_stories}]")["text"]
    tokens = []
    for story in stories:
        tokens += enc.encode_ordinary(story) + [enc.eot_token]   # eot = end of story
    return torch.tensor(tokens)

train_data = load_tokens("train", 100_000)
val_data   = load_tokens("validation", 5_000)
print("train tokens:", len(train_data), " val tokens:", len(val_data))

def get_batch(data, gen):
    # pick BATCH random starting points; y is x shifted by one token (the "next token")
    starts = torch.randint(len(data) - BLOCK - 1, (BATCH,), generator=gen)
    x = torch.stack([data[s : s + BLOCK] for s in starts])
    y = torch.stack([data[s + 1 : s + BLOCK + 1] for s in starts])
    return x.to(device), y.to(device)

## Step 3: The model (the important part)

`Attention` is the only part the paper changes. The `gate` setting picks where the gate goes:

| gate | where | code |
|---|---|---|
| `"none"` | no gate (baseline) | – |
| `"g1"` | after attention, before `W_o` (**paper's best**) | `y = y * g` |
| `"g2"` | after the value projection | `v = v * g` |
| `"g5"` | after the output projection `W_o` | `out = out * g` |

The gate is **head-specific**: after the heads are joined, columns 0–63 belong to head 1, columns 64–127 to head 2, and so on, so each head is scaled by its own gate values.
The gate is computed from `x`, the **current token**, so every token decides for itself how much of each head to let through.

In [ ]:
class Attention(nn.Module):
    def __init__(self, gate):
        super().__init__()
        self.gate = gate
        self.W_q = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_k = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_v = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_o = nn.Linear(D_MODEL, D_MODEL, bias=False)
        if gate != "none":
            self.W_g = nn.Linear(D_MODEL, D_MODEL, bias=False)   # gate weights

    def forward(self, x):
        B, T, D = x.shape
        H, d = N_HEADS, D // N_HEADS

        q, k, v = self.W_q(x), self.W_k(x), self.W_v(x)

        if self.gate != "none":
            g = torch.sigmoid(self.W_g(x))          # gate values between 0 and 1
            self.last_gate = g.detach()
        if self.gate == "g2":
            v = v * g                                # G2: gate the values

        # split into heads: (B, T, D) -> (B, H, T, d)
        q = q.view(B, T, H, d).transpose(1, 2)
        k = k.view(B, T, H, d).transpose(1, 2)
        v = v.view(B, T, H, d).transpose(1, 2)

        # scaled dot-product attention: softmax(Q K^T / sqrt(d)) V
        scores = q @ k.transpose(-2, -1) / math.sqrt(d)
        future = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
        scores = scores.masked_fill(future, float("-inf"))      # can't look at future tokens
        attn = torch.softmax(scores, dim=-1)                     # each row sums to 1
        self.last_attn = attn.detach()                           # saved to measure the sink
        y = attn @ v

        y = y.transpose(1, 2).reshape(B, T, D)       # join the heads back together

        if self.gate == "g1":
            y = y * g                                # G1: gate the attention output
        out = self.W_o(y)
        if self.gate == "g5":
            out = out * g                            # G5: gate after W_o
        return out


class Block(nn.Module):
    def __init__(self, gate):
        super().__init__()
        self.ln1 = nn.LayerNorm(D_MODEL)
        self.attn = Attention(gate)
        self.ln2 = nn.LayerNorm(D_MODEL)
        self.mlp = nn.Sequential(nn.Linear(D_MODEL, 4 * D_MODEL), nn.GELU(), nn.Linear(4 * D_MODEL, D_MODEL))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))    # attention + residual connection
        x = x + self.mlp(self.ln2(x))     # feed-forward + residual connection
        return x


class GPT(nn.Module):
    def __init__(self, gate):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB, D_MODEL)    # token id -> vector
        self.pos_emb = nn.Embedding(BLOCK, D_MODEL)    # position -> vector
        self.blocks = nn.ModuleList([Block(gate) for _ in range(N_LAYERS)])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, VOCAB)          # vector -> score for every next token

    def forward(self, idx):
        T = idx.shape[1]
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))


for gate in ["none", "g1", "g2", "g5"]:
    n = sum(p.numel() for p in GPT(gate).parameters())
    print(f"{gate:5s} {n/1e6:.1f}M parameters")

## Step 4: Training and measurements

- **Perplexity** = exp(validation loss). Lower is better.
- **Sink score** = average attention that tokens give to the *first* token. A high value means an attention sink.
- **Gate mean** = average gate value. Small means sparse (most outputs are switched down).
- **Spikes** = steps where the loss jumps 20% above its recent average (instability).

Every run uses the same seed and the same batches, so the only difference is the gate.

In [ ]:
def train(gate, lr):
    torch.manual_seed(0)                         # same starting weights
    gen = torch.Generator().manual_seed(0)       # same batches for every run
    model = GPT(gate).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for step in range(STEPS):
        for group in opt.param_groups:
            group["lr"] = lr * min(1, (step + 1) / WARMUP)   # warm-up

        x, y = get_batch(train_data, gen)
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB), y.view(-1))

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())
        if not math.isfinite(loss.item()):
            print(f"  {gate} lr={lr}: DIVERGED at step {step}")
            break
        if step % 500 == 0:
            print(f"  {gate} lr={lr} step {step} loss {loss.item():.3f}")
    return model, losses


@torch.no_grad()
def evaluate(model):
    model.eval()
    gen = torch.Generator().manual_seed(123)
    losses, sinks, gates = [], [], []
    for _ in range(20):
        x, y = get_batch(val_data, gen)
        logits = model(x)
        losses.append(F.cross_entropy(logits.view(-1, VOCAB), y.view(-1)).item())
        for block in model.blocks:
            attn = block.attn.last_attn                 # (B, H, T, T)
            sinks.append(attn[:, :, 1:, 0].mean().item())   # attention paid to token 0
            if block.attn.gate != "none":
                gates.append(block.attn.last_gate.mean().item())
    model.train()
    return {"perplexity": math.exp(np.mean(losses)),
            "sink_score": np.mean(sinks),
            "gate_mean": np.mean(gates) if gates else None}


def count_spikes(losses):
    return sum(1 for i in range(WARMUP + 50, len(losses))
               if losses[i] > 1.2 * np.mean(losses[i - 50 : i]))

## Step 5: Experiment 1 — compare gate positions

In [ ]:
results = {}
for gate in ["none", "g1", "g2", "g5"]:
    model, losses = train(gate, LR)
    results[gate] = {**evaluate(model), "spikes": count_spikes(losses), "losses": losses}
    print(gate, {k: v for k, v in results[gate].items() if k != "losses"})

## Step 6: Experiment 2 — stability at high learning rate (baseline vs G1)

In [ ]:
stability = {}
for lr in HIGH_LRS:
    for gate in ["none", "g1"]:
        model, losses = train(gate, lr)
        diverged = not math.isfinite(losses[-1])
        stability[f"{gate} lr={lr}"] = {
            "diverged": diverged,
            "spikes": count_spikes(losses),
            "perplexity": None if diverged else evaluate(model)["perplexity"],
            "losses": losses}

## Step 7: Results tables and plots

In [ ]:
print("Experiment 1: gate positions (lr = %g)" % LR)
print(f"{'gate':6s} {'perplexity':>11s} {'sink score':>11s} {'gate mean':>10s} {'spikes':>7s}")
for gate, r in results.items():
    gm = "-" if r["gate_mean"] is None else f"{r['gate_mean']:.3f}"
    print(f"{gate:6s} {r['perplexity']:11.2f} {r['sink_score']:11.3f} {gm:>10s} {r['spikes']:7d}")

print("\nExperiment 2: high learning rate")
for name, r in stability.items():
    ppl = "-" if r["perplexity"] is None else f"{r['perplexity']:.2f}"
    print(f"{name:16s} diverged={r['diverged']!s:5s} spikes={r['spikes']:3d} perplexity={ppl}")

json.dump({"exp1": results, "exp2": stability}, open("results.json", "w"))

# Plot 1: training loss for each gate position
plt.figure(figsize=(7, 4))
for gate, r in results.items():
    plt.plot(np.convolve(r["losses"], np.ones(50) / 50, mode="valid"), label=gate)
plt.ylim(top=5); plt.xlabel("step"); plt.ylabel("training loss"); plt.legend()
plt.title("Gate positions"); plt.savefig("plot_gate_positions.png", dpi=200); plt.show()

# Plot 2: attention-sink score
plt.figure(figsize=(5, 4))
plt.bar(results.keys(), [r["sink_score"] for r in results.values()])
plt.ylabel("attention on first token"); plt.title("Attention-sink score")
plt.savefig("plot_sink.png", dpi=200); plt.show()

# Plot 3: high learning rate
plt.figure(figsize=(7, 4))
for name, r in stability.items():
    plt.plot(r["losses"], lw=0.7, label=name)
plt.ylim(0, 10); plt.xlabel("step"); plt.ylabel("training loss"); plt.legend()
plt.title("Stability at high learning rate"); plt.savefig("plot_stability.png", dpi=200); plt.show()

## Notes for the report

- The paper uses 1.7B–15B parameter models trained on trillions of tokens. This is a ~30M parameter model trained on ~8M tokens, so differences will be smaller.
- Training windows start at random places in the text, so "token 0" is the first token of the window, not a special start token.
- G1 here is elementwise and head-specific, which is the paper's main version.